# Robot Synthetic Data Generation Workshop

## Background & Motivation

**The data scarcity problem** is the fundamental bottleneck in robot learning. Unlike language models trained on internet-scale text, or vision models trained on billions of images, robot learning requires physical interaction with the real world — expensive, slow, and hard to scale. A single demonstration episode might take 30 seconds, but robust policy training demands thousands of such episodes.

**Synthetic Data Generation (SDG)** addresses this by using physics simulation to generate unlimited expert demonstrations automatically. Among the four main SDG pathways — (A) Physics Simulation, (B) Video Reconstruction, (C) World Models, (D) Data Augmentation — **physics simulation** offers ground-truth state/action data, unlimited scale, domain randomization, and safe exploration.

This workshop implements a complete **Sim → Train → Eval** pipeline on AMD GPUs (ROCm) using the [Genesis](https://genesis-embodied-ai.github.io/) simulation engine and the [LeRobot](https://github.com/huggingface/lerobot) framework:

```
                 Pre-generated on RDNA4, published on HuggingFace
┌──────────────────────────┐     ┌─────────────────────┐     ┌──────────────────────┐
│ HF dataset (kitchen+wrist)│─▶  │ 02_train_vla.py      │ ─▶ │ 04_eval_custom_scene │
│ 100 ep · 13.5k frames     │    │ SmolVLA fine-tune    │    │ Closed-loop eval     │
│ 2 cams: up + wrist        │    │ freeze vision enc,   │    │ success rate + video │
│ AV1 video                 │    │ train expert + proj  │    │ render → VLA → PD    │
└──────────────────────────┘     └─────────────────────┘     └──────────────────────┘

     optional ────┐
     02_gen_data_custom_scene.py (regenerate data — prefers RDNA4 for 3-4× speedup)
```

**Workshop routing**: this notebook runs on a **CDNA3 (MI300 series)** node for training and evaluation. Data generation is pre-done on RDNA4 and pulled from HuggingFace — Section 1 includes a 2-episode demo for illustration, but you do not regenerate 100 episodes during the workshop. For benchmark-quality eval numbers, an RDNA4 node is preferred (MI300 falls back to CPU rasterization and introduces a ~20 pt success-rate bias; see Section 3).

## Data Flow

```
Genesis Scene                    LeRobot Dataset                SmolVLA
┌──────────────┐                ┌──────────────┐              ┌──────────────┐
│ Franka Panda │                │ observation   │              │ Vision       │
│ Red Cube     │──IK plan──────▶│  .state [9D]  │──train──────▶│ Encoder      │
│ 2 Cameras:   │   joint lerp   │  .images.up   │              │ (frozen)     │
│   up + wrist │   render       │  .images.side │              │              │
│ Physics sim  │                │  (= wrist cam)│              │ Expert       │
│ (Genesis)    │                │ action [9D]   │              │ Layers       │
└──────────────┘                │ task (text)   │              │ (trainable)  │
                                └──────────────┘              └──────────────┘
```

> Note: in the kitchen+wrist dataset the tensor key `observation.images.side` stores the **wrist (eye-in-hand) camera**, not a world-fixed side view.

## Workshop Steps

| Step | Script | Description |
|------|--------|-------------|
| **0** | Setup | Install deps, check GPU, download scene assets, cache HF dataset |
| **1a** | `01_gen_data.py` | Flat-scene data-gen demo (2 ep) — illustrates the IK pipeline on the simplest scene |
| **1b** | `02_gen_data_custom_scene.py --camera-layout up_wrist` | Kitchen + wrist data-gen demo (2 ep) — same recipe used for the HF training set |
| **1c** | HuggingFace | Load and visualize the full 100-episode kitchen+wrist dataset we actually train on |
| **2** | `02_train_vla.py` | Fine-tune SmolVLA (~450M params) on the HF kitchen+wrist dataset |
| **3** | `04_eval_custom_scene.py --camera-layout up_wrist` | Closed-loop evaluation in the kitchen scene |

---
## 0. Environment Setup

This section configures the runtime environment for the workshop:

- **Python imports** and output directory setup
- **GPU detection** — verifies CUDA/ROCm availability and VRAM capacity
- **Dependency installation** — `genesis-world` (physics sim), `lerobot` (dataset + models), `transformers` (SmolVLA backbone)
- **Scene asset download** — kitchen GLB meshes for the custom-scene pipeline

> **Hardware**: verified on **CDNA3 (AMD Instinct MI300 series, ROCm 6.x)** — the workshop primary target — and on **RDNA4 (Radeon AI PRO R9700, ROCm 7.2)**. Genesis uses the Taichi backend and runs natively on ROCm (no CUDA required). On MI300, Genesis rendering falls back to CPU llvmpipe (no graphics driver for Vulkan); on RDNA4 it uses GPU radeonsi — this is why data generation and evaluation favor RDNA4 when available (see `README.md` → Rendering Backend).

In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path
from IPython.display import display, Image, HTML, Video
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
matplotlib.rcParams['figure.dpi'] = 120

os.environ['HF_HUB_OFFLINE'] = '1'

WORKSHOP_DIR = Path(os.getcwd())
SCRIPTS_DIR = WORKSHOP_DIR / 'scripts'
OUTPUT_DIR = Path('/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Workshop dir: {WORKSHOP_DIR}')
print(f'Output dir:   {OUTPUT_DIR}')
print(f'Python:       {sys.executable}')

### 0.1 GPU & ROCm Check

In [ ]:
import torch

print(f'PyTorch:  {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / 1024**3
        print(f'  GPU[{i}]: {props.name}  VRAM: {vram_gb:.1f} GB')
else:
    print('WARNING: No GPU detected.')

!rocm-smi --showuse 2>/dev/null | head -20 || echo 'rocm-smi not available'

### 0.2 Install Dependencies

In [ ]:
!pip install -q lerobot==0.4.4 git+https://github.com/Genesis-Embodied-AI/Genesis.git@main \
  transformers accelerate safetensors matplotlib Pillow 2>&1 | tail -5

try:
    import torchcodec  # noqa: F401
    print(f'torchcodec OK: {torchcodec.__version__}')
except Exception as e:
    print(f'[warning] torchcodec import failed: {e}\n'
          '  The HF video dataset (Section 1c) and --num-workers 4 training (Section 2) '
          'will crash.\n  Build it from source now — see README Step 4b (CPU-only build).')

In [ ]:
import lerobot
print(f'lerobot: {lerobot.__version__}')

try:
    import genesis as gs
    print(f'genesis: {gs.__version__}')
except Exception as e:
    print(f'genesis import: {e}')

### 0.3 Download Kitchen Scene Assets (for custom-scene steps)

In [ ]:
!python {SCRIPTS_DIR}/00_download_kitchen.py --mesh-only

In [ ]:
asset_dir = WORKSHOP_DIR / 'assets' / 'rustic_kitchen'
if asset_dir.exists():
    for f in sorted(asset_dir.iterdir()):
        size_mb = f.stat().st_size / 1024**2
        print(f'  {f.name:40s}  {size_mb:>8.1f} MB')
else:
    print(f'Asset dir not found: {asset_dir}')

---
## 1. Synthetic Data Generation (Demo)

> ⚠️ **Demo only — you do not regenerate the full training set here.** The workshop's 100-episode kitchen+wrist dataset is pre-generated on RDNA4 and pulled directly from HuggingFace in Section 1c. The two generation cells below each run only 2 episodes, to illustrate the IK pipeline and the resulting on-disk LeRobot layout. On MI300 each episode takes ~28 s (CPU rasterization); on RDNA4 ~6–8 s.

This step uses **inverse kinematics (IK) planning** inside the Genesis physics simulator to generate expert-quality pick-cube trajectories. The IK planner produces a sequence of task-space waypoints (hover → descend → grasp → lift), solves for joint angles, and interpolates between them to create smooth trajectories.

### Why IK-based SDG?

Among the data collection paradigms — IK open-loop, IK closed-loop, and RL rollout — each trades off smoothness, recovery capability, and implementation complexity differently:

| Paradigm | Logic | Smoothness | Recovery |
|----------|-------|-----------|----------|
| **IK Open-loop** | One-shot IK solve → joint-space interpolation | Excellent | None |
| **IK Closed-loop** | Re-solve IK from current config each step | Good | Implicit |
| **RL Rollout** | Policy infers action from observation | Variable | Explicit |

This workshop uses **IK open-loop** for simplicity, producing near-perfect expert trajectories (≈100% success rate). While open-loop data lacks recovery signals, it provides a clean starting point for VLA post-training.

### Configuration

- **Robot**: Franka Panda (9 DOF: 7 arm joints + 2 finger joints)
- **Task**: Pick up a red cube from randomized XY positions
- **Trajectory**: `HOME → hover → descend → grasp → lift` (IK-planned, joint-space interpolated)
- **Cameras (main path)**: `up` (overhead) + `side` = wrist-mounted eye-in-hand camera (`--camera-layout up_wrist`), 640×480 RGB
- **Output**: LeRobot-format dataset with `observation.state` (9D), `action` (9D), `observation.images.{up,side}`

### 1a. Flat Scene Demo (2 episodes)

The baseline scene uses a flat ground plane with the Franka Panda and a red cube. The cube's XY position is randomized within the robot's reachable workspace for each episode. This is the simplest visual environment and is kept here as a reference to show that the IK pipeline itself is scene-agnostic. This is **not** the workshop training dataset — it ships with the legacy `up + world-side` camera layout and is shown only for illustration.

In [ ]:
N_EPISODES_FLAT = 2
REPO_ID_FLAT = 'local/workshop-flat-demo'

!python {SCRIPTS_DIR}/01_gen_data.py \
    --n-episodes {N_EPISODES_FLAT} \
    --repo-id {REPO_ID_FLAT} \
    --save {OUTPUT_DIR} \
    --seed 42 \
    --no-videos \
    --task 'Pick up the red cube.'

### 1b. Kitchen + Wrist Demo (2 episodes — workshop main-path recipe)

This demo uses the **same command and camera layout** that produced the 100-episode HuggingFace dataset used in training (Section 2). Two independent upgrades over 1a:

1. **Photo-realistic 3D kitchen scene** (GLB meshes from [World Labs Marble](https://marble.worldlabs.ai/)) instead of the flat plane — closes the visual domain gap. The scene uses a dual-mesh approach: HQ visual mesh (`rustic_kitchen.glb`) for rendering, simplified collider mesh for physics. The robot is placed at a configurable **anchor point** (`floor_origin`) and cube randomization happens in the robot-local coordinate frame.
2. **Eye-in-hand wrist camera** (`--camera-layout up_wrist`) in place of the world-fixed side camera. The tensor key `observation.images.side` now stores the wrist view — this is a deliberate schema choice so training code remains identical across layouts. Do not mix this dataset with `up+world-side` data: key names collide but semantics differ.

The full 100-episode dataset we actually train on was generated on RDNA4 with exactly this recipe (see Section 1c).

In [ ]:
N_EPISODES_KITCHEN = 2
REPO_ID_KITCHEN = 'local/workshop-kitchen-wrist-demo'

!python {SCRIPTS_DIR}/02_gen_data_custom_scene.py \
    --scene rustic_kitchen \
    --anchor floor_origin \
    --camera-layout up_wrist \
    --n-episodes {N_EPISODES_KITCHEN} \
    --repo-id {REPO_ID_KITCHEN} \
    --save {OUTPUT_DIR} \
    --seed 42 \
    --no-videos \
    --task 'Pick up the red cube.'

### 1c. Load the training dataset (from HuggingFace)

Now switch to the **actual training dataset** — 100 episodes of kitchen + wrist, pre-generated on RDNA4 and published at [`lidavidsh/franka-pick-kitchen-up-wrist-100ep-genesis`](https://huggingface.co/datasets/lidavidsh/franka-pick-kitchen-up-wrist-100ep-genesis). This is what Section 2 trains on.

A LeRobot dataset stores data in three directories:

```
dataset_root/
├── meta/           # info.json (feature defs, fps), stats.json (min/max/mean/std)
├── data/           # Parquet files: state, action, timestamps per frame
└── videos/         # MP4 (AV1) per camera: observation.images.up/, observation.images.side/
```

Below we load the HF dataset, print its schema, and visualize:
1. **Camera views** — sample frames from episode 0 (note: `observation.images.side` is the wrist camera)
2. **Joint trajectories** — `observation.state` vs `action` for each DOF over time
3. **Cube XY scatter** — spawn positions colored by success/failure (from the dataset's `episode_labels.json` if present)

In [ ]:
try:
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
except ImportError:
    from lerobot.datasets.lerobot_dataset import LeRobotDataset

os.environ.pop('HF_HUB_OFFLINE', None)

TRAIN_REPO = 'lidavidsh/franka-pick-kitchen-up-wrist-100ep-genesis'
ds = LeRobotDataset(TRAIN_REPO)
print(f'Dataset: {TRAIN_REPO}')
print(f'  Episodes:     {ds.num_episodes}')
print(f'  Total frames: {len(ds)}')
print(f'  FPS:          {ds.fps}')
print(f'  Features:     {list(ds.features.keys())}')

sample = ds[0]
print(f'\nSample keys: {list(sample.keys())}')
for k, v in sample.items():
    if hasattr(v, 'shape'):
        print(f'  {k}: shape={v.shape} dtype={v.dtype}')
    elif isinstance(v, str):
        print(f'  {k}: "{v}"')
    else:
        print(f'  {k}: {type(v).__name__} = {v}')

#### Camera Views

The plot below shows 6 evenly-spaced frames from episode 0 for each camera. For the workshop dataset (`kitchen + up_wrist` layout):

- **`observation.images.up`** — overhead world-fixed view (top row)
- **`observation.images.side`** — **wrist / eye-in-hand** camera mounted on the Franka hand link (bottom row). It follows the gripper through the approach-grasp-lift motion.

These are the RGB images that serve as visual input to the VLA model during training and closed-loop evaluation. The wrist view gives the policy local geometric detail at grasp time that an overhead-only view cannot provide.

**Sample wrist frames** (kitchen scene, 5 keyframes showing approach → descend → grasp → lift → sustain):

| up | wrist |
|:---:|:---:|
| ![up f033](images/kitchen_wrist/ep0_f033_up.png) | ![wrist f033](images/kitchen_wrist/ep0_f033_wrist.png) |
| ![up f067](images/kitchen_wrist/ep0_f067_up.png) | ![wrist f067](images/kitchen_wrist/ep0_f067_wrist.png) |
| ![up f100](images/kitchen_wrist/ep0_f100_up.png) | ![wrist f100](images/kitchen_wrist/ep0_f100_wrist.png) |

In [ ]:
# Visualize camera images from a sample episode
from PIL import Image as PILImage

def get_episode_range(dataset, ep_idx):
    if hasattr(dataset, 'episode_data_index'):
        start = dataset.episode_data_index['from'][ep_idx].item()
        end = dataset.episode_data_index['to'][ep_idx].item()
    else:
        fpe = len(dataset) // dataset.num_episodes
        start, end = ep_idx * fpe, (ep_idx + 1) * fpe
    return start, end

def show_episode_frames(dataset, ep_idx=0, n_frames=6):
    start, end = get_episode_range(dataset, ep_idx)
    indices = np.linspace(start, end - 1, n_frames, dtype=int)
    img_keys = [k for k in dataset.features if 'images' in k]
    if not img_keys:
        print('No image features found')
        return

    fig, axes = plt.subplots(len(img_keys), n_frames,
                             figsize=(3 * n_frames, 3 * len(img_keys)))
    if len(img_keys) == 1:
        axes = axes[np.newaxis, :]

    for row, key in enumerate(img_keys):
        for col, idx in enumerate(indices):
            sample = dataset[int(idx)]
            img = sample[key]
            if hasattr(img, 'numpy'):
                img = img.numpy()
            if img.ndim == 3 and img.shape[0] in (1, 3):
                img = np.transpose(img, (1, 2, 0))
            if img.dtype != np.uint8:
                if img.max() <= 1.0:
                    img = (img * 255).astype(np.uint8)
                else:
                    img = img.astype(np.uint8)
            axes[row, col].imshow(img)
            axes[row, col].set_title(f'frame {idx - start}', fontsize=8)
            axes[row, col].axis('off')
        axes[row, 0].set_ylabel(key.split('.')[-1], fontsize=10)

    fig.suptitle(f'Episode {ep_idx} Camera Views', fontsize=12)
    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR / f'ep{ep_idx}_camera_views.png'), bbox_inches='tight')
    plt.show()
    print(f'Saved: {OUTPUT_DIR / f"ep{ep_idx}_camera_views.png"}')

show_episode_frames(ds, ep_idx=0)

#### Joint Trajectories

The 3x3 grid below plots `observation.state` (blue) vs `action` (orange) for each of the 9 DOFs across episode 0. Joints `j1`–`j7` are the arm joints; `f1`–`f2` are the finger (gripper) joints.

- **state** = actual joint position read from the simulator (where the robot *is*)
- **action** = commanded target joint position (where we want the robot to *go*)

In IK-planned open-loop trajectories, the action–state gap is typically very small because the PD controller closely tracks the commanded positions. Large gaps or jitter here would indicate tracking issues or poor trajectory quality.

**Sample output** — notice the smooth IK-interpolated curves and the gripper close/open transitions (f1, f2):

![Episode 0 Joint Trajectories — state (blue) vs action (orange) for 9 DOFs](images/ep0_joint_trajectory.png)

In [ ]:
# Plot joint trajectories for episode 0
def plot_joint_trajectory(dataset, ep_idx=0):
    start, end = get_episode_range(dataset, ep_idx)
    states, actions = [], []
    for i in range(start, end):
        s = dataset[i]
        states.append(s['observation.state'].numpy())
        actions.append(s['action'].numpy())

    states = np.array(states)
    actions = np.array(actions)
    joint_names = ['j1','j2','j3','j4','j5','j6','j7','f1','f2']

    fig, axes = plt.subplots(3, 3, figsize=(14, 8), sharex=True)
    for i, ax in enumerate(axes.flat):
        if i >= states.shape[1]:
            ax.set_visible(False)
            continue
        ax.plot(states[:, i], label='state', linewidth=1)
        ax.plot(actions[:, i], label='action', linewidth=1, alpha=0.7)
        name = joint_names[i] if i < len(joint_names) else f'dim{i}'
        ax.set_title(name, fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    fig.suptitle(f'Episode {ep_idx} Joint Trajectories (state vs action)', fontsize=12)
    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR / f'ep{ep_idx}_joint_trajectory.png'), bbox_inches='tight')
    plt.show()

plot_joint_trajectory(ds, ep_idx=0)

#### Cube Spawn Position Scatter

The scatter plot below shows the XY spawn positions for all episodes in both the flat and kitchen scenes. Green circles = successful grasps, red crosses = failures. With IK-planned expert trajectories, we expect ≈100% success rate across the workspace. The distribution of spawn positions shows the range of randomization applied to the cube placement.

**Sample output** — kitchen scene, 100% success rate:

![Kitchen scene cube scatter — 100% SR](images/cube_scatter_kitchen.png)

In [ ]:
# Cube XY scatter: show all episode spawn positions + success/fail
def plot_cube_scatter(gen_dir):
    labels_path = gen_dir / 'episode_labels.json'
    summary_path = gen_dir / 'gen_summary.json'
    if not labels_path.exists():
        print(f'Labels not found: {labels_path}')
        return
    labels = json.loads(labels_path.read_text())
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}

    fig, ax = plt.subplots(figsize=(6, 5))
    for ep in labels:
        xy = ep.get('cube_xy') or ep.get('cube_world_xy') or ep.get('cube_local_dxdy')
        if xy is None:
            continue
        color = '#2ecc71' if ep['success'] else '#e74c3c'
        marker = 'o' if ep['success'] else 'x'
        ax.scatter(xy[0], xy[1], c=color, marker=marker, s=60, edgecolors='k', linewidths=0.5)

    ax.set_xlabel('X'); ax.set_ylabel('Y')
    sr = summary.get('success_rate', 0)
    ax.set_title(f'Cube Spawn Positions \u2014 SR: {sr:.0%}')
    ax.grid(True, alpha=0.3); ax.set_aspect('equal')
    plt.tight_layout()
    out = gen_dir / 'cube_xy_scatter.png'
    fig.savefig(str(out), bbox_inches='tight')
    plt.show()

print('=== Flat Scene ===')
plot_cube_scatter(OUTPUT_DIR / 'franka_gen_pick')
print('\n=== Kitchen Scene ===')
plot_cube_scatter(OUTPUT_DIR / 'custom_scene_gen')

---
## 2. SmolVLA Post-Training

### What is SmolVLA?

[SmolVLA](https://huggingface.co/blog/smolvla) is a **Vision-Language-Action** model that combines a visual encoder (SigLIP), a language model (SmolLM2), and an action expert to predict robot actions from images, proprioceptive state, and a natural-language task description.

| Component | Details |
|-----------|---------|
| **Architecture** | VLM (SigLIP + SmolLM2) + Action Expert |
| **Inputs** | images (up + side) + state (9D) + language task |
| **Output** | action chunk (50 × 9D joint targets) |
| **Total params** | ~450M |
| **Trainable params** | ~5M (expert only, vision encoder frozen) |

### Why SmolVLA for SDG?

Compared to simpler policies (MLP BC, ACT), SmolVLA brings:
- **Language conditioning** — the task instruction "Pick up the red cube" guides behavior
- **Pretrained vision** — SigLIP encoder provides strong visual features without training from scratch
- **Efficient fine-tuning** — only the action expert layers are trained, keeping VRAM under 4 GB

### Training Setup

We fine-tune `lerobot/smolvla_base` on the HuggingFace `kitchen + up_wrist` dataset (Section 1c) with:
- `chunk_size=50`, `n_action_steps=50`
- AdamW optimizer, `freeze_vision_encoder=True`
- Training only the expert layers + state projection head

**Default recipe** (recommended on both CDNA3 and RDNA4): `--batch-size 4 --num-workers 4`. `02_train_vla.py` automatically enables AMP BF16 when CUDA is available and relies on PyTorch's default SDPA dispatch, which picks the AOTriton flash kernel on AMD GPUs — no extra flags required. Together these give ~8× speedup over the old FP32 + math-SDPA baseline on RDNA4 (~0.09 s/step) and ~1.4× on CDNA3 (~0.14 s/step). See Reference Results in README for detail.

> **Note**: SmolVLA post-training requires specific `lerobot` versions for full compatibility. If you encounter `dataclass` errors with `lerobot>=0.5.0`, check the [LeRobot releases](https://github.com/huggingface/lerobot/releases) for a compatible version.

In [ ]:
N_TRAIN_STEPS = 4000
BATCH_SIZE = 4
SAVE_DIR = OUTPUT_DIR / 'outputs' / 'workshop_smolvla_kitchen_wrist'

!python {SCRIPTS_DIR}/02_train_vla.py \
    --dataset-id {TRAIN_REPO} \
    --n-steps {N_TRAIN_STEPS} \
    --batch-size {BATCH_SIZE} \
    --num-workers 4 \
    --save-dir {SAVE_DIR}

### 2.1 Training Loss Curve

The loss curve shows the MSE loss between predicted and ground-truth action chunks over training steps. The gradient norm plot (right) monitors training stability — spikes may indicate learning rate issues or data batch outliers.

**Important caveat**: A decreasing loss does *not* guarantee a working policy. As observed in our experiments with SmolVLA, the model can achieve low training loss by overfitting to specific training images while failing to generalize in closed-loop evaluation, where rendered images differ from the training distribution (especially with `freeze_vision_encoder=True`).

In [ ]:
def plot_loss_curve(save_dir):
    metrics_path = save_dir / 'train_metrics.json'
    summary_path = save_dir / 'train_summary.json'
    if not metrics_path.exists():
        print(f'Metrics not found: {metrics_path} (training may not have completed)')
        return

    metrics = json.loads(metrics_path.read_text())
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}

    steps = [m['step'] for m in metrics]
    losses = [m['loss'] for m in metrics]
    grad_norms = [m['grad_norm'] for m in metrics]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(steps, losses, linewidth=0.8, alpha=0.5, label='per-step')
    window = max(1, len(losses) // 20)
    if len(losses) > window:
        smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
        ax1.plot(steps[window-1:], smoothed, linewidth=2, color='#e74c3c', label=f'smooth(w={window})')
    ax1.set_xlabel('Step'); ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(steps, grad_norms, linewidth=0.8, alpha=0.6, color='#3498db')
    ax2.set_xlabel('Step'); ax2.set_ylabel('Grad Norm')
    ax2.set_title('Gradient Norm'); ax2.grid(True, alpha=0.3)

    info = ''
    if summary:
        info = (f"loss: {summary.get('loss_start','?'):.4f} -> {summary.get('loss_end','?'):.4f}  "
                f"time: {summary.get('total_time_s',0):.0f}s  VRAM: {summary.get('peak_vram_mb',0):.0f}MB")
    fig.suptitle(f'SmolVLA Training  |  {info}', fontsize=10)
    plt.tight_layout()
    fig.savefig(str(save_dir / 'loss_curve.png'), bbox_inches='tight')
    plt.show()

plot_loss_curve(SAVE_DIR)

---
## 3. Simulation Evaluation

### Open-loop Replay vs Closed-loop Evaluation

It is critical to distinguish between **replay** (open-loop) and **closed-loop** evaluation:

```
Open-loop Replay:                    Closed-loop Eval:
  dataset.action[t] → PD control      render → VLA inference → PD control
  No perception, no model              Full perception + model loop
  Tests: physics + control pipeline    Tests: end-to-end policy capability
```

| | Replay (sanity check) | Closed-loop (real eval) |
|---|---|---|
| **Action source** | Pre-recorded dataset | Model real-time inference |
| **Perception** | Not used | Render images + read joint state each step |
| **Tests** | Genesis physics + PD control + success criteria | End-to-end policy performance |

### Closed-loop Evaluation Loop

At each simulation step, the evaluation script runs:

1. **Render** `up` + `side` camera images from current scene state
2. **Read** current joint angles as `observation.state`
3. **Infer** action chunk via SmolVLA (images + state + task text)
4. **Post-process** predicted actions (denormalize to joint degrees)
5. **Execute** first action via PD position control
6. **Step** Genesis physics
7. **Check** success criteria: cube lifted ≥2cm and sustained ≥8 frames

### Success Metrics

| Metric | Definition |
|--------|-----------|
| **Grasp Success Rate** | `successful_episodes / total_episodes` |
| **Lift Height** | Maximum cube Z displacement during episode |
| **Sustain Frames** | Consecutive frames with cube above threshold |

### ⚠️ CPU-render evaluation bias (MI300)

MI300 does not expose a graphics driver for Vulkan, so Genesis falls back to CPU (llvmpipe) rendering for the camera images the policy sees. The CPU and GPU rasterizers produce visually different frames, and a policy trained on GPU-rendered data and evaluated with CPU rendering has a systematic **~20 pt lower success rate**. Reference numbers on the kitchen+wrist main path (see README Appendix A):

| train stack \ eval render | MI300 CPU (llvmpipe) | RDNA4 GPU (radeonsi) |
|---|:---:|:---:|
| CDNA3 train | **25 %** pooled | 45 % pooled |
| RDNA4 train | — (not tested) | **48 %** pooled |

**What this means for running this section on MI300**: expect ~25 % pooled over a few eval seeds, not a bug. For benchmark-quality numbers rerun this section on an RDNA4 node (drop `--render-cpu`). The eval cell below passes `--render-cpu` explicitly because that is the only path that works on an MI300 host; on RDNA4 you would omit it and Genesis will auto-select radeonsi.

> **Note**: Eval requires a trained checkpoint from Step 2. If training was skipped or incomplete, this section will be bypassed.

In [ ]:
CHECKPOINT = SAVE_DIR / 'final'
EVAL_UNSEEN_DIR = OUTPUT_DIR / 'eval_kitchen_wrist'

if CHECKPOINT.exists():
    !python {SCRIPTS_DIR}/04_eval_custom_scene.py \
        --policy-type smolvla \
        --checkpoint {CHECKPOINT} \
        --dataset-id {TRAIN_REPO} \
        --scene rustic_kitchen \
        --anchor floor_origin \
        --camera-layout up_wrist \
        --render-cpu \
        --n-episodes 20 --max-steps 150 --seed 99 \
        --record-video \
        --save {EVAL_UNSEEN_DIR}
else:
    print(f'No checkpoint at {CHECKPOINT} - skipping eval (training may not have completed)')

In [ ]:
# Display eval results and videos (if eval completed)
eval_summary_path = EVAL_UNSEEN_DIR / 'eval_summary.json'
if eval_summary_path.exists():
    summary = json.loads(eval_summary_path.read_text())
    print('=== Evaluation Results ===')
    for k, v in summary.items():
        print(f'  {k}: {v}')

    eval_videos = sorted(EVAL_UNSEEN_DIR.rglob('*.mp4'))
    if eval_videos:
        print(f'\nEval videos ({len(eval_videos)}):')
        for v in eval_videos[:3]:
            print(f'  {v.name}')
            try:
                display(Video(str(v), embed=True, width=640))
            except Exception as e:
                print(f'  (cannot embed video: {e})')
else:
    print('No eval summary found — training/eval may not have completed.')

---
## 4. Summary & Artifacts

This section collects and displays all artifacts generated during the workshop run:
- **PNG images** — camera views, joint trajectories, cube spawn scatter plots
- **MP4 videos** — per-episode evaluation rollouts (if eval completed)
- **JSON summaries** — data generation statistics, training metrics, evaluation results

### What We Accomplished

| Step | What | Key Output |
|------|------|-----------|
| **Data Gen (Flat)** | 10 episodes, Franka pick-cube on flat plane | LeRobot dataset + scatter plot |
| **Data Gen (Kitchen)** | 10 episodes, same task in 3D kitchen scene | LeRobot dataset + scatter plot |
| **Visualization** | Camera views, joint trajectories, spawn positions | Inline plots + saved PNGs |
| **VLA Training** | SmolVLA fine-tune on flat-scene data | Checkpoint + loss curve |
| **Evaluation** | Closed-loop SmolVLA in Genesis | Success rate + videos |

In [ ]:
print('=== Generated Artifacts ===')
pngs = sorted(OUTPUT_DIR.rglob('*.png'))
mp4s = sorted(OUTPUT_DIR.rglob('*.mp4'))
jsons = sorted(OUTPUT_DIR.rglob('*.json'))

print(f'\nPNG images ({len(pngs)}):')
for p in pngs:
    print(f'  {p.relative_to(OUTPUT_DIR)}  ({p.stat().st_size/1024:.0f} KB)')

print(f'\nMP4 videos ({len(mp4s)}):')
for p in mp4s:
    print(f'  {p.relative_to(OUTPUT_DIR)}  ({p.stat().st_size/1024**2:.1f} MB)')

print(f'\nJSON summaries ({len(jsons)}):')
for p in jsons:
    print(f'  {p.relative_to(OUTPUT_DIR)}')

In [ ]:
# Display all generated PNGs inline
for p in pngs:
    print(f'\n--- {p.relative_to(OUTPUT_DIR)} ---')
    display(Image(filename=str(p), width=700))

In [ ]:
# Show data generation summaries
for gen_name in ['franka_gen_pick', 'custom_scene_gen']:
    sp = OUTPUT_DIR / gen_name / 'gen_summary.json'
    if sp.exists():
        summary = json.loads(sp.read_text())
        print(f'\n=== {gen_name} ===')
        for k, v in summary.items():
            if k not in ('success_episode_ids', 'failure_episode_ids'):
                print(f'  {k}: {v}')
        print(f'  success_rate: {summary.get("success_rate", "N/A")}')

In [ ]:
print('\nWorkshop pipeline complete!')
print(f'All outputs saved to: {OUTPUT_DIR}')

---
## References & Further Reading

### Frameworks
- [LeRobot](https://github.com/huggingface/lerobot) — Robot learning framework by HuggingFace (dataset format, policies, training)
- [Genesis](https://genesis-embodied-ai.github.io/) — GPU-accelerated physics simulation engine (ROCm compatible via Taichi)
- [SmolVLA](https://huggingface.co/blog/smolvla) — Vision-Language-Action model for robot control

### SDG & Imitation Learning
- [DART](https://proceedings.mlr.press/v78/laskey17a.html) (CoRL 2017) — Noise injection with closed-loop supervisor for reducing covariate shift
- [MimicGen](https://mimicgen.github.io/) (CoRL 2023) — Few demonstrations → large-scale data via motion replanning
- [Domain Randomization for Sim-to-Real Transfer](https://arxiv.org/abs/1703.06907) — Visual/physical randomization for transfer

### VLA Research (2025–2026)
- [DiffRL→VLA](https://arxiv.org/abs/2509.19752) — Diffusion RL for smoother rollout trajectories
- [SimpleVLA-RL](https://arxiv.org/abs/2509.09674) (ICLR 2026) — Direct RL fine-tuning of VLA models
- [RL-Co](https://arxiv.org/abs/2602.12628) — SFT warm-start + sim RL fine-tune + real data anti-forgetting
- [DLR](https://arxiv.org/abs/2511.19528) — Diverse RL trajectories for VLA pretraining

### AMD ROCm
- [AMD ROCm Documentation](https://rocm.docs.amd.com/)
- [PyTorch ROCm Installation](https://pytorch.org/get-started/locally/)
- [World Labs Marble](https://marble.worldlabs.ai/) — 3D scene generation for custom simulation environments